<a href="https://colab.research.google.com/github/ced-sys/AI-N-ML/blob/main/Old_Assyrian_Gemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
from pathlib import Path
from tqdm.auto import tqdm
from typing import List, Dict, Tuple
import warnings
import pickle
warnings.filterwarnings('ignore')

In [ ]:
from transformers import(
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import evaluate
from sklearn.model_selection import train_test_split

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
  print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
def upload_and_unzip(extract_to: Path=Path('/content/data'))-> Path:
  try:
    from google.colab import files
    IN_COLAB=True
  except ImportError:
    IN_COLAB=False

  extract_to.mkdir(exist_ok=True, parents=True)

  if IN_COLAB:
    uploaded=files.upload()

    zip_filename=list(uploaded.keys())[0]
    print(f"\nUploaded: {zip_filename}")

    print(f"\nExtracting to {extract_to}...")
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
      zip_ref.extractall(extract_to)

    extracted_files=list(extract_to.glob('*'))
    print(f"\nExtracted {len(extracted_files)} items:")
    for f in sorted(extracted_files)[:10]:
      print(f"  -{f.name}")
    if len(extracted_files)>10:
      print(f"  ... and {len(extracted_files)-10} more")

    Path(zip_filename).unlink()
    print(f"\nCleaned up {zip_filename}")

  else:
    zip_path_input=input("\nEnter path to your zip file (or press enter to skip):").strip()

    if zip_path_input and Path(zip_path_input).exists():
      zip_path=Path(zip_path_input)
      print(f"\nExtracting {zip_path} to {extract_to}...")
      with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
      print("Extraction complete!")
    else:
      print("\Skipping extraction. Make sure your data is in the DATA_DIR!")

  required_files=['train.csv', 'test_csv', 'published_texts.csv', 'publications.csv']
  found_files=[]
  missing_files=[]

  for filename in required_files:
    file_path=extract_to / filename
    if file_path.exists():
      size_mb=file_path.stat().st_size /(1024*1024)
      found_files.append(filename)
      print(f"{filename:30s} ({size_mb:>6.1f} MB)")
    else:
      missing_files.append(filename)
      print(f" {filename:30s}(MISSING)")

  print(f"Found: {len(found_files)}/{len(required_files)} required files")

  if missing_files:
    print(f"\n WARNING: Missing files: {', '.join(missing_files)}")
  else:
    print("\nAll required files found")


  return extract_to


In [ ]:
DATA_DIR=Path('/content/data')
OUTPUT_DIR=Path('/content/drive/MyDrive/old_assyrian_models')
KAGGLE_EXPORT_DIR=Path('/content/drive/MyDrive/old_assyrian_kaggle')

MODEL_NAME="google/gemma-2-2b-it"
MAX_LENGTH=512

NUM_EPOCHS=3
BATCH_SIZE=4
GRADIENT_ACCUMULATION_STEPS=4
LEARNING_RATE=2e-4
LORA_RANK=16
LORA_ALPHA=32

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
KAGGLE_EXPORT_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
class AkkadianNormalizer:

  def__init__(self):
    self.vowel_map={
        '\u00e1':'a2',
        '\u00e0':'a3',
        '\u00e2':'a2',
        '\u00e9':'e2',
        '\u00e8':'e3',
        '\u00ea':'e2',
        '\u00ed':'i2',
        '\u00ec':'i3',
        '\u00ee':'i2',
        '\u00fa':'u2',
        '\u00f9':'u3',
        '\u00fb':'u2',
    }

    self.consonant_map={
        '\u0161':'sz',
        '\u0160':'SZ',
        '\u1e63':'S',
        '\u1e62':'S',
        '\u1e6d':'t',
        '\u1e6c':'T',
        '\u1e2b':'h',
        '\u1e2a':'H',
    }